In [1]:
%%capture
%pip install -U browser-use playwright python-dotenv langchain-google-genai pydantic
%playwright install chromium --with-deps

In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your GOOGLE_API_KEY: ")

In [3]:
import asyncio
import json
import re
from pathlib import Path

from pydantic import BaseModel, Field, SecretStr
from langchain_google_genai import ChatGoogleGenerativeAI
from browser_use import Agent, Browser, Tools, ActionResult

START_URL = "https://www.zehitomo.com/"
SERVICE_URL = "https://www.zehitomo.com/service-list"
AIRCON_URL = (
    "https://www.zehitomo.com/home/cleaning/"
    "electronics-cleaning/air-conditioner-cleaner"
)
POSTAL_CODE = "182-0004"
OUTPUT_FILE = "zehitomo_contractors.json"


class Contractor(BaseModel):
    contractor_name: str = Field(
        description="Contractor/company name at the top of the chat box"
    )
    location: str = Field(
        description="Location name shown in the contractor details/map area"
    )
    phone_number: str = Field(
        default="",
        description="Phone number found in chat messages; empty string if absent",
    )
    email: str = Field(
        default="",
        description="Email address found in chat messages; empty string if absent",
    )


class ScrapingResult(BaseModel):
    postal_code: str
    service: str
    contractors: list[Contractor]

In [4]:
tools = Tools()


@tools.action(
    "Wait for the user to manually complete login on Zehitomo, then continue."
)
async def wait_for_manual_login() -> ActionResult:
    input(
        "\n>>> A Chrome window will open.\n"
        ">>> Please click 「ログイン」 and complete login manually.\n"
        ">>> After login, press ENTER here to continue...\n"
    )
    return ActionResult(extracted_content="Manual login completed.")


@tools.action("Wait 60 seconds before scraping the contractor list.")
async def wait_one_minute() -> ActionResult:
    print("Waiting 60 seconds for the contractor page to load...")
    await asyncio.sleep(60)
    return ActionResult(extracted_content="Finished waiting 60 seconds.")


@tools.action(
    "Extract phone number and email from the currently open chat page."
)
async def extract_current_contractor(browser_session) -> ActionResult:
    page = await browser_session.get_current_page()
    text = await page.inner_text("body")

    email_match = re.search(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text
    )
    phone_match = re.search(
        r"(?<!\d)(?:0\d{1,4}[-−]\d{1,4}[-−]\d{3,4}|0\d{9,10})(?!\d)",
        text,
    )

    return ActionResult(
        extracted_content=json.dumps(
            {
                "phone_number": phone_match.group(0).replace("−", "-") if phone_match else "",
                "email": email_match.group(0) if email_match else "",
                "page_text_preview": text[:3000],
            },
            ensure_ascii=False,
        )
    )

INFO     [service] Using anonymized telemetry, see https://docs.browser-use.com/development/monitoring/telemetry.


In [5]:
TASK = f"""
You are automating Zehitomo using a real visible Chrome browser.

Follow these steps exactly:

1. Go to {START_URL}.

2. Call `wait_for_manual_login`.
   - The user will manually click 「ログイン」 and complete login.
   - Do not enter credentials.
   - Continue only after the user confirms login.

3. Reload {START_URL}.

4. Click 「サービス一覧」 in the navbar and go to {SERVICE_URL}.

5. Click 「エアコンクリーニング」.
   It should lead to:
   {AIRCON_URL}

6. Click 「戸建タイプ」.

7. In the modal:
   - Repeatedly select the FIRST visible radio-button option.
   - Click 「次へ」.
   - Continue until 「スキップ」 appears.

8. When 「スキップ」 appears, always click it.

9. Enter `{POSTAL_CODE}` into the postal-code search field.
10. Click 「プロを見る」.

11. If 「前回の依頼を確認する」 and 「新しく依頼を出す」 appear,
    click 「前回の依頼を確認する」.
12. Call `wait_one_minute`.

13. On the contractor page, find the floating scrollable contractor list on the left.

14. For every contractor:
    - Click the contractor name.
    - In the chat box, read the contractor/company name at the top.
    - Click that contractor name to open the details panel.
    - Extract the location shown in the map/details area.
    - Extract phone number and email from chat messages if present.
    - Use empty strings for missing phone/email.
    - Call `extract_current_contractor` for phone/email support.
    - Return to the contractor list and process the next contractor.

15. Do not invent data.
16. Return structured JSON for all contractors found for postal code `{POSTAL_CODE}`.
"""

In [7]:
from browser_use import ChatGoogle

llm = ChatGoogle(
    model="gemini-2.5-flash",
    api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,
)

browser = Browser(
    headless=False,
    keep_alive=True,
    window_size={"width": 1440, "height": 900},
)

agent = Agent(
    task=TASK,
    llm=llm,
    browser=browser,
    tools=tools,
    structured_output=ScrapingResult,
    max_actions_per_step=5,
)

try:
    history = await agent.run(max_steps=250)

    result = history.structured_output

    if result is None:
        final = history.final_result()
        if not final:
            raise RuntimeError("Gemini did not return a structured result.")
        result = ScrapingResult.model_validate_json(final)

    data = result.model_dump(mode="json", exclude_none=True)

    Path(OUTPUT_FILE).write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"Saved {len(data['contractors'])} contractors to {OUTPUT_FILE}")
    print(json.dumps(data, ensure_ascii=False, indent=2))

finally:
    await browser.kill()

INFO     [Agent] 🎯 Task: 
You are automating Zehitomo using a real visible Chrome browser.

Follow these steps exactly:

1. Go to https://www.zehitomo.com/.

2. Call `wait_for_manual_login`.
   - The user will manually click 「ログイン」 and complete login.
   - Do not enter credentials.
   - Continue only after the user confirms login.

3. Reload https://www.zehitomo.com/.

4. Click 「サービス一覧」 in the navbar and go to https://www.zehitomo.com/service-list.

5. Click 「エアコンクリーニング」.
   It should lead to:
   https://www.zehitomo.com/home/cleaning/electronics-cleaning/air-conditioner-cleaner

6. Click 「戸建タイプ」.

7. In the modal:
   - Repeatedly select the FIRST visible radio-button option.
   - Click 「次へ」.
   - Continue until 「スキップ」 appears.

8. When 「スキップ」 appears, always click it.

9. Enter `182-0004` into the postal-code search field.
10. Click 「プロを見る」.

11. If 「前回の依頼を確認する」 and 「新しく依頼を出す」 appear,
    click 「前回の依頼を確認する」.
12. Call `wait_one_minute`.

13. On the contractor page, find the floating sc

RuntimeError: Gemini did not return a structured result.